In [6]:
from kaggle_secrets import UserSecretsClient
secret_label = "Token"
token = UserSecretsClient().get_secret(secret_label)

! git clone https://{token}@github.com/ZhengJinFa168/RecSys-Challenge-2025.git

fatal: destination path 'RecSys-Challenge-2025' already exists and is not an empty directory.


In [7]:
! pip install PyGithub requests

In [8]:
! pip install implicit

In [11]:
cd RecSys-Challenge-2025/

/kaggle/working/RecSys-Challenge-2025


In [12]:
import numpy as np
import pandas as pd
import scipy.sparse as sps
import matplotlib.pyplot as pyplot
from Evaluation.Evaluator import EvaluatorHoldout
from Data_manager.split_functions.split_train_validation_random_holdout import split_train_in_two_percentage_global_sample
from Recommenders.EASE_R.EASE_R_Recommender import EASE_R_Recommender
from Recommenders.GraphBased.RP3betaRecommender import RP3betaRecommender
from Recommenders.GraphBased.P3alphaRecommender import P3alphaRecommender
from Recommenders.KNN.ItemKNNCFRecommender import ItemKNNCFRecommender
from Recommenders.SLIM.SLIMElasticNetRecommender import SLIMElasticNetRecommender
from Recommenders.Similarity.Compute_Similarity_Python import Compute_Similarity_Python
from helping_methods import toOutput, evaluate_algorithm
import optuna
import time
import optuna.visualization as vis
from Recommenders.MatrixFactorization.PureSVDRecommender import PureSVDRecommender

In [13]:
    URM_all_dataframe = pd.read_csv('data/data_train.csv')
    users_to_test = pd.read_csv('data/data_target_users_test.csv')
    user_id_list = users_to_test['user_id'].tolist()

    URM_all_dataframe.columns = ["UserID", "ItemID"]
    URM_all_dataframe['hasInteraction'] = 1
    URM_all = sps.csr_matrix((URM_all_dataframe['hasInteraction'].values,
                              (URM_all_dataframe['UserID'].values, URM_all_dataframe['ItemID'].values)))

    URM_train, URM_test = split_train_in_two_percentage_global_sample(URM_all, train_percentage=0.80)

    evaluator_test = EvaluatorHoldout(URM_test, cutoff_list=[20])

EvaluatorHoldout: Ignoring 27 ( 0.1%) Users that have less than 1 test interactions


In [17]:
def objective(trial):
    
    recommender = PureSVDRecommender(URM_train)

    # 1.1. Suggest hyperparameters
    num_factors = trial.suggest_int('num_factors', 10, 1000)

    recommender.fit( num_factors=num_factors, random_seed = 1)
    
    score, _ = evaluator_test.evaluateRecommender(recommender)
    recall = score['RECALL']
    # 1.4. Return the score (Optuna minimizes by default, so we return 1 - accuracy)
    return recall # Or use direction='maximize' in create_study

def main():


    start_time = time.time()

    study = optuna.create_study(
        direction='maximize',
        study_name='PureSVDRecommender1',
        storage='sqlite:///OptunaStudies/PureSVDRecommender.db',  # This saves to a file
    )
    study.optimize(objective, n_trials=50)  # Run 50 trials
    # 3. Print the results
    print("Number of finished trials:", len(study.trials))
    print("Best trial:")
    trial = study.best_trial

    print(f"  Value (1 - Accuracy): {trial.value}")
    print("  Params: ")
    for key, value in trial.params.items():
        print(f"    {key}: {value}")

    # To get the best accuracy:
    best_accuracy = 1.0 - trial.value
    print(f"\nBest Cross-Validated Accuracy: {best_accuracy:.4f}")

    # Plot the optimization history
    vis.plot_optimization_history(study).show()

    # Plot the parameter importances
    vis.plot_param_importances(study).show()

    # Plot a slice of the parameters vs the objective value
    vis.plot_slice(study).show()

    # Plot the parallel coordinates
    vis.plot_parallel_coordinate(study).show()

    outputFile = "output.csv"

    end_time = time.time()

if __name__ == "__main__":
    main()

[I 2025-12-03 13:47:31,186] A new study created in RDB with name: PureSVDRecommender1


PureSVDRecommender: Computing SVD decomposition...
PureSVDRecommender: Computing SVD decomposition... done in 19.76 sec
EvaluatorHoldout: Processed 27068 (100.0%) in 18.25 sec. Users per second: 1483


/usr/local/lib/python3.11/dist-packages/optuna/study/_tell.py:66: FutureWarning:

Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead

/usr/local/lib/python3.11/dist-packages/optuna/study/_tell.py:70: FutureWarning:

Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead

/usr/local/lib/python3.11/dist-packages/optuna/study/_tell.py:167: FutureWarning:

Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead

[I 2025-12-03 13:48:09,349] Trial 0 finished with value: 0.10392467488303482 and parameters: {'num_factors': 997}. Best is trial 0 with value: 0.10392467488303482.


PureSVDRecommender: Computing SVD decomposition...
PureSVDRecommender: Computing SVD decomposition... done in 3.56 sec
EvaluatorHoldout: Processed 27068 (100.0%) in 14.79 sec. Users per second: 1830


/usr/local/lib/python3.11/dist-packages/optuna/study/_tell.py:66: FutureWarning:

Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead

/usr/local/lib/python3.11/dist-packages/optuna/study/_tell.py:70: FutureWarning:

Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead

/usr/local/lib/python3.11/dist-packages/optuna/study/_tell.py:167: FutureWarning:

Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead

[I 2025-12-03 13:48:27,833] Trial 1 finished with value: 0.18851612136898155 and parameters: {'num_factors': 142}. Best is trial 1 with value: 0.18851612136898155.


...

PureSVDRecommender: Computing SVD decomposition...
PureSVDRecommender: Computing SVD decomposition... done in 1.53 sec
EvaluatorHoldout: Processed 27068 (100.0%) in 14.54 sec. Users per second: 1862


/usr/local/lib/python3.11/dist-packages/optuna/study/_tell.py:66: FutureWarning:

Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead

/usr/local/lib/python3.11/dist-packages/optuna/study/_tell.py:70: FutureWarning:

Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead

/usr/local/lib/python3.11/dist-packages/optuna/study/_tell.py:167: FutureWarning:

Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead

[I 2025-12-03 14:05:33,911] Trial 48 finished with value: 0.20137070566830548 and parameters: {'num_factors': 47}. Best is trial 18 with value: 0.20217057026168989.


PureSVDRecommender: Computing SVD decomposition...
PureSVDRecommender: Computing SVD decomposition... done in 1.43 sec
EvaluatorHoldout: Processed 27068 (100.0%) in 14.47 sec. Users per second: 1871


/usr/local/lib/python3.11/dist-packages/optuna/study/_tell.py:66: FutureWarning:

Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead

/usr/local/lib/python3.11/dist-packages/optuna/study/_tell.py:70: FutureWarning:

Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead

/usr/local/lib/python3.11/dist-packages/optuna/study/_tell.py:167: FutureWarning:

Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead

[I 2025-12-03 14:05:49,936] Trial 49 finished with value: 0.20087810121153424 and parameters: {'num_factors': 43}. Best is trial 18 with value: 0.20217057026168989.


Number of finished trials: 50
Best trial:
  Value (1 - Accuracy): 0.20217057026168989
  Params: 
    num_factors: 34

Best Cross-Validated Accuracy: 0.7978
